In [1]:
from sklearn.ensemble import RandomForestClassifier
import mysql.connector
import pandas as pd
import joblib

def fetch_aligned_data(symbol='eth', name='ethereum'):
    print(f"📡 Fetching aligned data for {symbol}...")
    try:
        DB_CONFIG = {'host':'localhost','user':'root','password':'','database':'crypto_radar_db'}
        conn = mysql.connector.connect(**DB_CONFIG)

        # We fetch sentiment for the SPECIFIC coin AND BITCOIN separately
        query = f"""
        WITH all_sentiment AS (
            SELECT timestamp, senti_score, weight, asset FROM posts_logs
            WHERE asset IN ('{symbol}', '{name}', 'bitcoin', 'market', 'crypto')
            UNION ALL
            SELECT timestamp, senti_score, 1.0 as weight, asset FROM news_logs
            WHERE asset IN ('{symbol}', '{name}', 'bitcoin', 'market', 'crypto')
        )
        SELECT
            p.timestamp,
            p.price_close as price,
            p.volume_usdt as volume,

            -- 1. LOCAL SENTIMENT (The Coin Itself)
            (SELECT AVG(senti_score * weight)
             FROM all_sentiment
             WHERE asset IN ('{symbol}', '{name}')
             AND timestamp BETWEEN DATE_SUB(p.timestamp, INTERVAL 4 HOUR) AND p.timestamp
            ) as sentiment_coin,

            -- 2. GLOBAL SENTIMENT (Bitcoin - The Market Leader)
            (SELECT AVG(senti_score * weight)
             FROM all_sentiment
             WHERE asset = 'bitcoin'
             AND timestamp BETWEEN DATE_SUB(p.timestamp, INTERVAL 4 HOUR) AND p.timestamp
            ) as sentiment_btc,

            -- 3. MACRO (DXY)
            (SELECT value FROM macro_indicators WHERE indicator_code = 'DXY' ORDER BY ABS(TIMESTAMPDIFF(SECOND, timestamp, p.timestamp)) ASC LIMIT 1) as dxy_index,

            LEAD(p.price_close, 4) OVER (ORDER BY p.timestamp) as target_next_price
        FROM coin_prices p
        WHERE p.asset = '{symbol}'
        ORDER BY p.timestamp ASC
        """

        df = pd.read_sql(query, conn)
        conn.close()

        # --- FEATURE ENGINEERING ---
        # Fill missing sentiment with "Neutral" (0.5)
        df['sentiment_coin'] = df['sentiment_coin'].fillna(0.5)
        df['sentiment_btc'] = df['sentiment_btc'].fillna(0.5) # Critical if BTC data is missing
        df['dxy_index'] = df['dxy_index'].ffill().bfill()

        df = df.dropna(subset=['target_next_price'])

        # Lags & Trends (Calculate for BOTH)
        df['sentiment_trend_coin'] = df['sentiment_coin'].rolling(window=3).mean()
        df['sentiment_trend_btc'] = df['sentiment_btc'].rolling(window=3).mean() # Bitcoin Trend

        df['price_change_pct'] = df['price'].pct_change()

        # Technicals
        delta = df['price'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))

        df['volatility'] = df['price'].pct_change().rolling(window=24).std()
        df['sma_50'] = df['price'].rolling(window=50).mean()
        df['dist_from_sma'] = (df['price'] / df['sma_50']) - 1

        df = df.dropna()

        # Noise Filter (> 0.25% move)
        threshold = 0.0025
        df['move_pct'] = (df['target_next_price'] - df['price']) / df['price']
        df_clean = df[abs(df['move_pct']) > threshold].copy()

        df_clean['target_class'] = (df_clean['move_pct'] > 0).astype(int)

        print(f"✅ Data Ready. Shape: {df_clean.shape}")
        return df_clean

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def train_model(df):
    # UPDATE FEATURES LIST
    features = [
        'volume', 'dxy_index', 'price_change_pct',
        'sentiment_coin', 'sentiment_trend_coin', # Local Mood
        'sentiment_btc', 'sentiment_trend_btc',   # Global Mood (The "Cheat Code")
        'rsi', 'volatility', 'dist_from_sma'
    ]

    # --- BALANCE DATA (Fixes "All Bearish" issue) ---
    # print(df['target_class'])
    bulls = df[df['target_class'] == 1]
    bears = df[df['target_class'] == 0]
    min_len = min(len(bulls), len(bears))

    # Force equal numbers
    df_balanced = pd.concat([
        bulls.sample(n=min_len, random_state=42),
        bears.sample(n=min_len, random_state=42)
    ])

    X = df_balanced[features]
    y = df_balanced['target_class']

    print(f"🧠 Training with {len(features)} features (Inc. BTC Sentiment)...")
    model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
    model.fit(X, y)

    # Save...
    joblib.dump(model, "../models/crypto_pulse_brain.pkl")
    print("💾 Saved Upgraded Model.")

if __name__ == "__main__":
    df = fetch_aligned_data("btc","bitcoin")
    train_model(df)


📡 Fetching aligned data for btc...


C:\Users\dhair\AppData\Local\Temp\ipykernel_9840\729795163.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Data Ready. Shape: (2107, 16)
🧠 Training with 10 features (Inc. BTC Sentiment)...
💾 Saved Upgraded Model.


btc,eth,sol,doge,xrp

MODEL_FILE = "../models/crypto_pulse_brain.pkl"
# --- Flask Initialization ---
app = Flask(__name__)
CORS(app)
# --- initialize the model ---
model = joblib.load("../models/crypto_pulse_brain.pkl"

In [6]:
try:
    print(f"🧠 Loading AI Model from: ../models/crypto_pulse_brain.pkl")
    model = joblib.load("../models/crypto_pulse_brain.pkl")
    print("✅ Model Active.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model = None
def fetch_live_features(symbol='eth',name='ethereum'):
    """
    Fetches the last 6 snapshots (24h) to calculate Lags (t-1, t-2, t-3)
    and Trends for the current moment.
    """
    try:
        conn = get_db_connection()

        # 1. SQL Query: Get Price, Volume, DXY, and Weighted Sentiment
        # We fetch LIMIT 10 to ensure we have enough history for lags/rolling
        query = f"""
        WITH all_sentiment AS (
            SELECT timestamp, senti_score, weight, asset FROM posts_logs WHERE asset IN ('{symbol}', '{name}', 'market', 'crypto')
            UNION ALL
            SELECT timestamp, senti_score, 1.0 as weight, asset FROM news_logs WHERE asset IN ('{symbol}', '{name}', 'market', 'crypto')
        )
        SELECT
            p.timestamp,
            p.price_close as price,
            p.volume_usdt as volume,

            -- Weighted Sentiment Calculation (Matches Training Logic)
            (SELECT AVG(senti_score * weight * CASE
                    WHEN asset = '{symbol}' OR asset = '' THEN 1.0
                    ELSE 0.5
                 END)
             FROM all_sentiment
             WHERE timestamp BETWEEN DATE_SUB(p.timestamp, INTERVAL 4 HOUR) AND p.timestamp
            ) as sentiment_current,

            -- DXY Lookup
            (SELECT value FROM macro_indicators
             WHERE indicator_code = 'DXY'
             ORDER BY ABS(TIMESTAMPDIFF(SECOND, timestamp, p.timestamp)) ASC LIMIT 1
            ) as dxy_index

        FROM coin_prices p
        WHERE p.asset = '{symbol}'
        ORDER BY p.timestamp DESC
        LIMIT 10
        """

        df = pd.read_sql(query, conn)
        conn.close()

        if len(df) < 5:
            return None # Not enough history to calculate lags

        # 2. Sort Chronologically (Oldest -> Newest) for calculation
        df = df.sort_values('timestamp').reset_index(drop=True)

        # 3. Clean Missing Data
        df['sentiment_current'] = df['sentiment_current'].fillna(0.5)
        df['dxy_index'] = df['dxy_index'].ffill().bfill()

        # 4. Engineer Features (Must match training columns EXACTLY)
        df['sentiment_lag_1'] = df['sentiment_current'].shift(1)
        df['sentiment_lag_2'] = df['sentiment_current'].shift(2)
        df['sentiment_lag_3'] = df['sentiment_current'].shift(3)

        df['sentiment_trend_12h'] = df['sentiment_current'].rolling(window=3).mean()
        df['sent_vol_interaction'] = df['sentiment_current'] * df['volume']
        df['sentiment_change'] = df['sentiment_current'] - df['sentiment_lag_1']
        df['price_change_pct'] = df['price'].pct_change()

        # 5. Extract the Final Row (The "Now" Snapshot)
        latest = df.iloc[[-1]].dropna()

        if latest.empty:
            return None

        return latest

    except Exception as e:
        print(f"Feature Engineering Error: {e}")
        return None

if model:
    live_df = fetch_live_features('btc')

    if live_df is not None:
        # Select features in the EXACT order of training
        features = [
                'volume', 'dxy_index', 'price_change_pct',
                'sentiment_current', 'sentiment_lag_1', 'sentiment_lag_2', 'sentiment_lag_3',
                'sentiment_trend_12h', 'sent_vol_interaction', 'sentiment_change'
        ]

        try:
            X_live = live_df[features]

            # Predict
            prediction = model.predict(X_live)[0]
            probs = model.predict_proba(X_live)[0]
            confidence = np.max(probs)
            print(prediction)
            # Signal Logic (Elite Threshold > 0.60)
            signal_text = "HOLD"
            if confidence > 0.60:
                signal_text = "STRONG BUY" if prediction == 1 else "SELL"
            elif confidence > 0.55:
                signal_text = "WEAK BUY" if prediction == 1 else "WEAK SELL"
            print("direction: " "BULLISH" if prediction == 1 else "BEARISH",
                "confidence: " + str(round(confidence * 100, 1)),
                "signal: " + signal_text,
                "sentiment_score: " + str(int(live_df['sentiment_current'].values[0] * 100)),
                "timestamp: " + str(live_df['timestamp'].values[0]))
        except Exception as e:
                print(f"error {str(e)}", "confidence:0")


🧠 Loading AI Model from: ../models/crypto_pulse_brain.pkl
✅ Model Active.
Feature Engineering Error: name 'get_db_connection' is not defined


In [7]:
import mysql.connector
import pandas as pd
import numpy as np
import joblib
import os
import sys

# --- CONFIGURATION ---
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '',
    'database': 'crypto_radar_db'
}


# Load Brain
try:
    print(f"🧠 Loading AI Model from: ../models/crypto_pulse_brain.pkl")
    model = joblib.load("../models/crypto_pulse_brain.pkl")
    print("✅ Model Active.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model = None

def get_db_connection():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_aligned_data(symbol='eth',name='ethereum'):
    print(f"📡 Fetching aligned data (2022-Present)...")
    try:
        conn = mysql.connector.connect(**DB_CONFIG)

        # SQL Query (Same as before, fetches ALL history)
        sentiment_filter = f"asset IN ('{symbol}', '{name}', 'market', 'crypto')"
        query = f"""
        WITH all_sentiment AS (
            SELECT timestamp, senti_score, weight, asset FROM posts_logs WHERE {sentiment_filter}
            UNION ALL
            SELECT timestamp, senti_score, 1.0 as weight, asset FROM news_logs WHERE {sentiment_filter}
        )
        SELECT
            p.timestamp,
            p.price_close as price,
            p.volume_usdt as volume,
            (SELECT AVG(senti_score * weight * CASE WHEN asset = '{symbol}' OR asset = 'ethereum' THEN 1.0 ELSE 0.5 END)
             FROM all_sentiment
             WHERE timestamp BETWEEN DATE_SUB(p.timestamp, INTERVAL 4 HOUR) AND p.timestamp
            ) as sentiment_current,
            (SELECT value FROM macro_indicators WHERE indicator_code = 'DXY' ORDER BY ABS(TIMESTAMPDIFF(SECOND, timestamp, p.timestamp)) ASC LIMIT 1) as dxy_index,
            LEAD(p.price_close, 3) OVER (ORDER BY p.timestamp) as target_next_price
        FROM coin_prices p
        WHERE p.asset = '{symbol}'
        ORDER BY p.timestamp ASC
        """

        df = pd.read_sql(query, conn)
        conn.close()

        # --- 1. CLEANING ---
        df['sentiment_current'] = df['sentiment_current'].fillna(0.5)
        df['dxy_index'] = df['dxy_index'].ffill().bfill()
        df = df.dropna(subset=['target_next_price'])

        # --- 2. EXISTING FEATURES ---
        df['sentiment_lag_1'] = df['sentiment_current'].shift(1)
        df['sentiment_lag_2'] = df['sentiment_current'].shift(2)
        df['sentiment_trend_12h'] = df['sentiment_current'].rolling(window=3).mean()
        df['price_change_pct'] = df['price'].pct_change()

        # --- 3. NEW SMART FEATURES (RSI, Volatility, SMA) ---

        # RSI (14 periods)
        delta = df['price'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))

        # Volatility (Standard Deviation of last 24h)
        df['volatility'] = df['price'].pct_change().rolling(window=24).std()

        # Distance from Moving Average (Trend Indicator)
        # (Price / 50-Hour Average) - 1. Positive = Above trend, Negative = Below
        df['sma_50'] = df['price'].rolling(window=50).mean()
        df['dist_from_sma'] = (df['price'] / df['sma_50']) - 1

        # Drop NaNs created by rolling windows
        df = df.dropna()

        # Target (1 = UP, 0 = DOWN)
        df['target_class'] = (df['target_next_price'] > df['price']).astype(int)

        print(f"✅ Data Loaded & Engineered. Shape: {df.shape}")
        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None


def print_terminal_report():
    if not model:
        print("❌ Model Offline. Cannot predict.")
        return

    print("\n📡 Fetching Live Market Data from MySQL...")
    live_df = fetch_live_features('btc')

    if live_df is not None:
        # Select features in the EXACT order of training
        features = [
            'volume', 'dxy_index', 'price_change_pct',
            'sentiment_current', 'sentiment_lag_1', 'sentiment_lag_2', 'sentiment_lag_3',
            'sentiment_trend_12h', 'sent_vol_interaction', 'sentiment_change'
        ]

        try:
            X_live = live_df[features]

            # Predict
            prediction = model.predict(X_live)[0]
            probs = model.predict_proba(X_live)[0]
            confidence = np.max(probs)

            # Context Data
            current_price = float(live_df['price'].values[0])
            sentiment_val = float(live_df['sentiment_current'].values[0])
            dxy_val = float(live_df['dxy_index'].values[0])
            timestamp = str(live_df['timestamp'].values[0])

            # Signal Logic
            signal_text = "HOLD"
            if confidence > 0.60:
                signal_text = "STRONG BUY" if prediction == 1 else "STRONG SELL"
            elif confidence > 0.55:
                signal_text = "WEAK BUY" if prediction == 1 else "WEAK SELL"

            # --- TARGET PRICE CALCULATION ---
            # Based on the 0.5% volatility threshold used during training
            volatility_factor = 0.005
            if prediction == 1:
                target_price = current_price * (1 + volatility_factor)
                move_str = f"+${(target_price - current_price):.2f} (+0.5%)"
            else:
                target_price = current_price * (1 - volatility_factor)
                move_str = f"-${(current_price - target_price):.2f} (-0.5%)"

            # --- TERMINAL OUTPUT ---
            # ANSI Colors: Green=\033[92m, Red=\033[91m, Reset=\033[0m
            c_dir = "\033[92m" if prediction == 1 else "\033[91m"
            c_rst = "\033[0m"

            print("\n" + "="*50)
            print(f"🔮 CRYPTO PULSE AI: LIVE FORECAST")
            print("="*50)
            print(f"📅 Timestamp:    {timestamp}")
            print(f"💰 Current BTC:  ${current_price:,.2f}")
            print(f"💵 DXY Index:    {dxy_val:.2f}")
            print(f"🗣️ Hype Score:   {sentiment_val:.4f} (0-1 Scale)")
            print("-" * 50)
            print(f"🤖 AI Prediction: {c_dir}{'BULLISH (UP)' if prediction == 1 else 'BEARISH (DOWN)'}{c_rst}")
            print(f"🎯 Target Price:  ${target_price:,.2f} ({move_str})")
            print(f"🛡️ Confidence:    {confidence:.2%}")
            print(f"🚦 Trade Signal:  {c_dir}{signal_text}{c_rst}")
            print("="*50 + "\n")

        except Exception as e:
            print(f"❌ Prediction failed: {str(e)}")
    else:
        print("❌ Not enough history (Need 12h of data in DB to run).")

if __name__ == '__main__':
    # Run directly in terminal
    print_terminal_report()

🧠 Loading AI Model from: ../models/crypto_pulse_brain.pkl
✅ Model Active.

📡 Fetching Live Market Data from MySQL...


C:\Users\dhair\AppData\Local\Temp\ipykernel_14576\3139985726.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


❌ Prediction failed: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- sent_vol_interaction
- sentiment_change
- sentiment_lag_3
Feature names seen at fit time, yet now missing:
- dist_from_sma
- rsi
- volatility

